In [1]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
embeddings.embed_query("Hello AI")

[-0.033388249576091766,
 0.03453972190618515,
 0.05947450175881386,
 0.05928609147667885,
 -0.06353534013032913,
 -0.06819584965705872,
 0.08823327720165253,
 0.03444086015224457,
 -0.03278521075844765,
 -0.015814999118447304,
 0.020981688052415848,
 -0.018340280279517174,
 -0.03983212634921074,
 -0.0804707482457161,
 -0.014469251036643982,
 0.033264849334955215,
 0.014259209856390953,
 -0.034050002694129944,
 -0.142915740609169,
 -0.02308330126106739,
 -0.021380094811320305,
 0.0026335834991186857,
 -0.04729274660348892,
 -0.010752721689641476,
 -0.06866800785064697,
 0.03112504631280899,
 0.0759459137916565,
 0.0011282932246103883,
 0.011631960049271584,
 -0.03603917732834816,
 0.04483765736222267,
 0.018390756100416183,
 0.12672796845436096,
 -0.0013598083751276135,
 0.008206670172512531,
 0.06909963488578796,
 -0.0807635560631752,
 -0.05841314047574997,
 0.05375446006655693,
 0.026227610185742378,
 -0.006828599609434605,
 -0.05635838210582733,
 0.0032929580193012953,
 -0.0725017562

In [4]:
len(embeddings.embed_query("Hello AI"))

384

In [ ]:
# Configuring the google embedding
from langchain_google_genai import GoogleGenerativeAIEmbeddings

os.environ['GOOGLE_API_KEY']=os.getenv("GOOGLE_API_KEY")
google_api_key=os.getenv('GOOGLE_API_KEY')
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [14]:
embeddings.embed_query("Hello AI")


[-0.027124856,
 0.028169623,
 -0.00781581,
 -0.0029762103,
 -0.00060313934,
 0.01986695,
 -0.0034572317,
 0.032462526,
 -0.0010989095,
 -0.051247932,
 -0.018915892,
 -0.009750103,
 -0.025037337,
 -0.00975446,
 0.012220629,
 -0.014473387,
 0.04482668,
 -0.006226926,
 -0.009329216,
 -0.002217293,
 -0.004965341,
 0.00902455,
 -0.013654741,
 0.026774528,
 -0.013447652,
 0.03259606,
 0.018450413,
 -0.014387199,
 -0.026691856,
 0.114021055,
 -0.015732974,
 -0.020877438,
 -0.008109539,
 -0.008938644,
 0.012272543,
 -0.018457945,
 -0.010577812,
 -0.02339095,
 0.0022810355,
 -0.008221659,
 0.020857066,
 0.030231897,
 0.034944393,
 0.0032693911,
 0.0007356355,
 -0.018876733,
 -0.028560907,
 -0.024729973,
 0.019272905,
 -0.005892965,
 -0.022996014,
 0.0050585903,
 -0.016044319,
 -0.013149028,
 -0.017556453,
 -0.0020654288,
 -0.01665227,
 0.0033709551,
 -0.026166067,
 0.004227843,
 -0.014302048,
 -0.0059348107,
 0.011485234,
 0.024174964,
 0.0048312414,
 0.0152847115,
 0.005070072,
 -0.022278033,


In [15]:
len(embeddings.embed_query("Hello AI"))

3072

### PINECONE

In [16]:
## to configure pinecone, we need a pinecone client
from pinecone import Pinecone

In [17]:
pinecone_api_key=os.getenv("PINECONE_API_KEY")

In [18]:
pc=Pinecone(api_key=pinecone_api_key)

In [19]:
from pinecone import ServerlessSpec # serverless means the server managed by the cloud provider, we don't have to manage it


In [46]:
index_name='agentic-ai-index'

In [47]:
pc.has_index(index_name)

False

In [48]:
## Creating Index
if not pc.has_index(index_name):
    pc.create_index(
    name=index_name,
    dimension=3072,
    metric="cosine",
    spec=ServerlessSpec(cloud='aws', region='us-east-1')
)

In [49]:
# Loading the index
index=pc.Index(index_name)

In [50]:
from langchain_pinecone import PineconeVectorStore


In [51]:
vector_store=PineconeVectorStore(index=index, embedding=embeddings)

In [52]:
result=vector_store.similarity_search("What is langchain?")

In [53]:
result

[]

In [54]:
from uuid import uuid4
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with langchain - come check it out!.",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! that was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!.",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to the fears of recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)



In [55]:
documents=[
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [56]:
documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with langchain - come check it out!.'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! that was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applicatio

In [57]:
## UUID - universal identification number, FAISS was creating uuid itself
uuids=[str(uuid4()) for _ in range(len(documents))]

In [58]:
uuids

['d5cf0478-8398-443e-ab3f-c63a23301c26',
 '2197ee41-1d2c-42f9-81b5-bfa40432ec80',
 '6386f666-71c0-4cfb-8bc1-053b8b0b0138',
 '5d7a19ca-592a-4548-a135-0ed69c71a27e',
 'c85ae8e0-50fa-49da-9fc8-0282f54749a6',
 'd63e4ae0-57d5-4094-ba15-ed796a72c886',
 '3bbc2204-e574-4de4-b540-766ab195f92c',
 'bb7f1eae-3a1f-4920-b2ec-cb8da6cbcc9d',
 'a87e26a1-77c0-459d-983c-d9a9add577a1',
 '940d902b-7214-4148-818e-69f3da6fc9be']

In [59]:
vector_store.add_documents(documents=documents, ids=uuids)

['d5cf0478-8398-443e-ab3f-c63a23301c26',
 '2197ee41-1d2c-42f9-81b5-bfa40432ec80',
 '6386f666-71c0-4cfb-8bc1-053b8b0b0138',
 '5d7a19ca-592a-4548-a135-0ed69c71a27e',
 'c85ae8e0-50fa-49da-9fc8-0282f54749a6',
 'd63e4ae0-57d5-4094-ba15-ed796a72c886',
 '3bbc2204-e574-4de4-b540-766ab195f92c',
 'bb7f1eae-3a1f-4920-b2ec-cb8da6cbcc9d',
 'a87e26a1-77c0-459d-983c-d9a9add577a1',
 '940d902b-7214-4148-818e-69f3da6fc9be']

In [62]:
result=vector_store.similarity_search("What is langchain?", k=2)

In [63]:
result

[Document(id='6386f666-71c0-4cfb-8bc1-053b8b0b0138', metadata={'source': 'tweet'}, page_content='Building an exciting new project with langchain - come check it out!.'),
 Document(id='bb7f1eae-3a1f-4920-b2ec-cb8da6cbcc9d', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!.')]

In [ ]:
result=vector_store.similarity_search("What is langchain?", filter={"source": "tweet"})
result

[Document(id='6386f666-71c0-4cfb-8bc1-053b8b0b0138', metadata={'source': 'tweet'}, page_content='Building an exciting new project with langchain - come check it out!.'),
 Document(id='bb7f1eae-3a1f-4920-b2ec-cb8da6cbcc9d', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!.'),
 Document(id='940d902b-7214-4148-818e-69f3da6fc9be', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :('),
 Document(id='d5cf0478-8398-443e-ab3f-c63a23301c26', metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.')]

In [ ]:
retriever=vector_store.as_retriever(
    # search_kwargs={"k": 3},
    score_threshold=0.99 # similarity score more than 99%
)

In [82]:
retriever.invoke("google")

[Document(id='940d902b-7214-4148-818e-69f3da6fc9be', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :('),
 Document(id='6386f666-71c0-4cfb-8bc1-053b8b0b0138', metadata={'source': 'tweet'}, page_content='Building an exciting new project with langchain - come check it out!.'),
 Document(id='bb7f1eae-3a1f-4920-b2ec-cb8da6cbcc9d', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!.'),
 Document(id='d63e4ae0-57d5-4094-ba15-ed796a72c886', metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.')]

In [83]:
from langchain_google_genai import ChatGoogleGenerativeAI
model=ChatGoogleGenerativeAI(model='gemini-1.5-flash')

In [84]:
from langsmith import Client

client = Client()
prompt = client.pull_prompt("rlm/rag-prompt")

In [85]:
import pprint
pprint.pprint(prompt.messages)

[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


In [87]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [88]:
def format_docs(docs):
    return "\n".join([d.page_content for d in docs])

In [90]:
rag_chain=(
    {"context":retriever | format_docs, "question": RunnablePassthrough(),}
    | prompt
    | model
    | StrOutputParser()
)
rag_chain.invoke("What is langchain?")

ChatGoogleGenerativeAIError: Error calling model 'gemini-1.5-flash' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

#### CUSTOM PROMPT

In [100]:
from langchain_core.prompts import PromptTemplate

In [101]:
prompt=PromptTemplate(
    template="""
    Answer the question based on the context provided.
    Context: {context}
    Question: {question}
    Answer:
    """,
    input_variables=["context", "question"]
    )

In [102]:
prompt.invoke({"context": "langchain is a framework for building stateful, agentic applications.", "question": "What is langchain?"})

StringPromptValue(text='\n    Answer the question based on the context provided.\n    Context: langchain is a framework for building stateful, agentic applications.\n    Question: What is langchain?\n    Answer:\n    ')

In [105]:
## Assignment:
# -> Complete the above remainingpart with the proper data & create proper RAG
# -> Take a pdf with text, image, table with atlkeast 200 pages, if chunking required, use semantic chunking do chunking then embedding
#1. fetch data from the pdf, store it in the vector DB (use any: mongodb, astradb, opensearch, milvus)
#2. Create a index with all 3 index mech(Flat, HNSW, IVF)
#3. Create a Retriever pipeline
#4. Check the retriever time, which one is fastest
#5. Print the accuracy score of every similarity search
#6. Perform the reranking either using BM25 or MMR
#7. Write prompt template
#8. Generate o/p through LLM
#9. render the o/p over the docx